In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ============================================================
# TEAM 3 - PHASE 5 INCREMENTAL GOLD JOB
#
# Processes only orders affected by the latest Silver batch,
# then MERGEs recomputed rows into Gold.
#
# Gold outputs:
#   1. order_line_fulfilment
#   2. order_service_performance
#   3. regional_backlog
# ============================================================

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------
dbutils.widgets.text("catalog", "dbwteam3")
dbutils.widgets.text("schema", "order_fulfillment")
dbutils.widgets.text("input_batch", "latest")
dbutils.widgets.text("as_of_time", "2026-10-01T00:00:00")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
INPUT_BATCH = dbutils.widgets.get("input_batch").strip().lower()
AS_OF_TIME = dbutils.widgets.get("as_of_time").strip()
BASE = f"{CATALOG}.{SCHEMA}"

ORDERS_TABLE = f"{BASE}.silver_orders"
ORDER_LINES_TABLE = f"{BASE}.silver_order_lines"
DELIVERY_EVENTS_TABLE = f"{BASE}.silver_delivery_events"
PHARMACIES_TABLE = f"{BASE}.silver_pharmacies"

GOLD_LINE_TABLE = f"{BASE}.order_line_fulfilment"
GOLD_ORDER_TABLE = f"{BASE}.order_service_performance"
GOLD_REGION_TABLE = f"{BASE}.regional_backlog"

required_tables = [
    ORDERS_TABLE, ORDER_LINES_TABLE, DELIVERY_EVENTS_TABLE, PHARMACIES_TABLE,
    GOLD_LINE_TABLE, GOLD_ORDER_TABLE, GOLD_REGION_TABLE,
]
for table in required_tables:
    if not spark.catalog.tableExists(table):
        raise ValueError(
            f"Required table is missing: {table}. "
            "Run the existing full-refresh Gold job once before switching to incremental Gold."
        )

# ------------------------------------------------------------
# 2. RESOLVE THE SILVER BATCH
# ------------------------------------------------------------
batch_union = None
for table in [ORDERS_TABLE, ORDER_LINES_TABLE, DELIVERY_EVENTS_TABLE]:
    df = (
        spark.table(table)
        .select(F.col("_batch_id").cast("string").alias("_batch_id"))
        .filter(F.col("_batch_id").rlike(r"^batch_[0-9]{3}$"))
        .distinct()
    )
    batch_union = df if batch_union is None else batch_union.unionByName(df)

available_batches = [r["_batch_id"] for r in batch_union.distinct().collect()]
if not available_batches:
    raise ValueError("No valid batch_NNN values found in Silver")

if INPUT_BATCH == "latest":
    BATCH_ID = max(available_batches, key=lambda x: int(x.split("_")[1]))
elif INPUT_BATCH in available_batches:
    BATCH_ID = INPUT_BATCH
else:
    raise ValueError(f"input_batch must be latest or an existing batch_NNN: {INPUT_BATCH}")

print("=" * 70)
print("PHASE 5 - INCREMENTAL GOLD JOB")
print("Catalog/schema :", BASE)
print("Silver batch   :", BATCH_ID)
print("Reporting as-of:", AS_OF_TIME)
print("=" * 70)

# ------------------------------------------------------------
# 3. IDENTIFY AFFECTED ORDERS ONLY
#
# A Gold order is affected if this Silver batch contains:
# - a new/changed order
# - a new order line
# - a new delivery event
# ------------------------------------------------------------
orders_batch = spark.table(ORDERS_TABLE).filter(F.col("_batch_id") == BATCH_ID)
lines_batch = spark.table(ORDER_LINES_TABLE).filter(F.col("_batch_id") == BATCH_ID)
events_batch = spark.table(DELIVERY_EVENTS_TABLE).filter(F.col("_batch_id") == BATCH_ID)

affected_order_ids = (
    orders_batch.select("order_id")
    .unionByName(lines_batch.select("order_id"))
    .unionByName(events_batch.select("order_id"))
    .filter(F.col("order_id").isNotNull())
    .distinct()
)

affected_order_count = affected_order_ids.count()
if affected_order_count == 0:
    print(f"No affected Silver orders for {BATCH_ID}; nothing to update in Gold.")
    dbutils.notebook.exit("NO_CHANGES")

print("Affected orders:", affected_order_count)
print("Batch Silver orders:", orders_batch.count())
print("Batch Silver order lines:", lines_batch.count())
print("Batch Silver delivery events:", events_batch.count())

# ------------------------------------------------------------
# 4. READ ONLY THE REQUIRED SILVER SCOPE
#
# We need historical Silver rows for affected orders because Gold
# metrics are cumulative. We do NOT rescan unrelated orders.
# ------------------------------------------------------------
orders_affected = (
    spark.table(ORDERS_TABLE)
    .join(affected_order_ids, "order_id", "inner")
)
order_lines_affected = (
    spark.table(ORDER_LINES_TABLE)
    .join(affected_order_ids, "order_id", "inner")
)
events_affected = (
    spark.table(DELIVERY_EVENTS_TABLE)
    .join(affected_order_ids, "order_id", "inner")
)
pharmacies = spark.table(PHARMACIES_TABLE).select("pharmacy_id", "region")

# ------------------------------------------------------------
# 5. RECOMPUTE LINE GOLD FOR AFFECTED ORDERS
# ------------------------------------------------------------
delivery_summary = (
    events_affected
    .groupBy("order_id", "line_id")
    .agg(
        F.sum("delivered_quantity").alias("uncapped_delivered_units"),
        F.max("delivery_time").alias("last_delivery_time"),
        F.max(
            F.when(F.col("delivered_quantity") > 0, F.col("delivery_time"))
        ).alias("last_positive_delivery_time"),
    )
)

delivery_window = (
    Window
    .partitionBy("order_id", "line_id")
    .orderBy("delivery_time", "event_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

event_progress = events_affected.withColumn(
    "cumulative_delivered_units",
    F.sum("delivered_quantity").over(delivery_window),
)

completion_times = (
    event_progress
    .join(
        order_lines_affected.select("order_id", "line_id", "ordered_quantity"),
        ["order_id", "line_id"],
        "inner",
    )
    .filter(F.col("cumulative_delivered_units") >= F.col("ordered_quantity"))
    .groupBy("order_id", "line_id")
    .agg(F.min("delivery_time").alias("completion_time"))
)

line_gold_delta = (
    order_lines_affected
    .join(delivery_summary, ["order_id", "line_id"], "left")
    .join(
        orders_affected.select(
            "order_id", "order_time", "promised_delivery_time", "order_status"
        ),
        "order_id",
        "left",
    )
    .join(completion_times, ["order_id", "line_id"], "left")
    .withColumn("uncapped_delivered_units", F.coalesce("uncapped_delivered_units", F.lit(0)))
    .withColumn(
        "capped_delivered_units",
        F.least(F.col("uncapped_delivered_units"), F.col("ordered_quantity")),
    )
    .withColumn(
        "open_units",
        F.greatest(F.lit(0), F.col("ordered_quantity") - F.col("capped_delivered_units")),
    )
    .withColumn(
        "fulfilment_percentage",
        F.round(F.col("capped_delivered_units") / F.col("ordered_quantity") * 100, 2),
    )
    .withColumn(
        "is_overdelivered",
        F.col("uncapped_delivered_units") > F.col("ordered_quantity"),
    )
    .withColumn(
        "backlog_age_days",
        F.when(
            F.col("open_units") > 0,
            F.greatest(
                F.lit(0),
                F.datediff(F.to_timestamp(F.lit(AS_OF_TIME)), F.col("order_time")),
            ),
        ),
    )
)

# Align columns exactly to existing Gold target.
line_target_cols = spark.table(GOLD_LINE_TABLE).columns
line_gold_delta = line_gold_delta.select(*line_target_cols)

# UPSERT line rows because existing line metrics can change when later deliveries arrive.
line_target = DeltaTable.forName(spark, GOLD_LINE_TABLE)
(
    line_target.alias("t")
    .merge(
        line_gold_delta.alias("s"),
        "t.order_id = s.order_id AND t.line_id = s.line_id",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("Merged:", GOLD_LINE_TABLE)

# ------------------------------------------------------------
# 6. RECOMPUTE ORDER GOLD FOR AFFECTED ORDERS
# ------------------------------------------------------------
line_order_metrics = (
    line_gold_delta
    .groupBy("order_id")
    .agg(
        F.sum("ordered_quantity").alias("total_ordered_units"),
        F.sum("capped_delivered_units").alias("total_capped_delivered_units"),
        F.sum("uncapped_delivered_units").alias("total_uncapped_delivered_units"),
        F.sum("open_units").alias("total_open_units"),
        F.min(F.when(F.col("open_units") > 0, 0).otherwise(1)).alias("all_lines_complete"),
        F.min(
            F.when(
                (F.col("open_units") == 0)
                & (F.col("completion_time") <= F.col("promised_delivery_time")),
                1,
            ).otherwise(0)
        ).alias("all_lines_on_time"),
        F.max(F.col("is_overdelivered").cast("int")).alias("has_overdelivery"),
    )
)

order_gold_delta = (
    orders_affected.select("order_id", "promised_delivery_time", "order_status")
    .join(line_order_metrics, "order_id", "left")
    .withColumn("total_ordered_units", F.coalesce("total_ordered_units", F.lit(0)))
    .withColumn("total_capped_delivered_units", F.coalesce("total_capped_delivered_units", F.lit(0)))
    .withColumn("total_uncapped_delivered_units", F.coalesce("total_uncapped_delivered_units", F.lit(0)))
    .withColumn("total_open_units", F.coalesce("total_open_units", F.lit(0)))
    .withColumn("all_lines_complete", F.coalesce("all_lines_complete", F.lit(0)))
    .withColumn("all_lines_on_time", F.coalesce("all_lines_on_time", F.lit(0)))
    .withColumn("has_overdelivery", F.coalesce("has_overdelivery", F.lit(0)))
    .withColumn(
        "order_fulfilment_percentage",
        F.when(
            F.col("total_ordered_units") > 0,
            F.round(F.col("total_capped_delivered_units") / F.col("total_ordered_units") * 100, 2),
        ).otherwise(F.lit(None).cast("double")),
    )
    .withColumn("is_cancelled", F.lower(F.trim("order_status")) == "cancelled")
    .withColumn(
        "is_promised_by_as_of",
        F.col("promised_delivery_time") <= F.to_timestamp(F.lit(AS_OF_TIME)),
    )
    .withColumn(
        "otif_denominator_eligible",
        (~F.col("is_cancelled")) & F.col("is_promised_by_as_of"),
    )
    .withColumn(
        "otif_pass",
        F.col("otif_denominator_eligible")
        & (F.col("total_ordered_units") > 0)
        & (F.col("all_lines_complete") == 1)
        & (F.col("all_lines_on_time") == 1),
    )
    .withColumn(
        "is_overdue",
        (~F.col("is_cancelled"))
        & F.col("is_promised_by_as_of")
        & ((F.col("total_open_units") > 0) | (F.col("total_ordered_units") == 0)),
    )
)

order_target_cols = spark.table(GOLD_ORDER_TABLE).columns
order_gold_delta = order_gold_delta.select(*order_target_cols)

order_target = DeltaTable.forName(spark, GOLD_ORDER_TABLE)
(
    order_target.alias("t")
    .merge(order_gold_delta.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("Merged:", GOLD_ORDER_TABLE)

# ------------------------------------------------------------
# 7. UPDATE REGIONAL BACKLOG FOR AFFECTED REGIONS
#
# Regional metrics depend on all orders in a region. We therefore
# identify only the regions touched by this batch, then reaggregate
# Gold order rows for those regions. Other regions are untouched.
# ------------------------------------------------------------
affected_orders_with_region = (
    orders_affected.select("order_id", "pharmacy_id")
    .join(pharmacies, "pharmacy_id", "left")
    .select("order_id", "region")
)

affected_regions = (
    affected_orders_with_region
    .select("region")
    .filter(F.col("region").isNotNull())
    .distinct()
)

# Map all Silver orders only for the affected region names, then attach persisted
# Gold order metrics. This preserves correct regional totals without rebuilding
# unrelated regions.
all_order_regions_for_affected_regions = (
    spark.table(ORDERS_TABLE)
    .select("order_id", "pharmacy_id")
    .join(pharmacies, "pharmacy_id", "left")
    .join(affected_regions, "region", "inner")
)

regional_gold_delta = (
    all_order_regions_for_affected_regions
    .join(spark.table(GOLD_ORDER_TABLE), "order_id", "inner")
    .groupBy("region")
    .agg(
        F.sum("total_open_units").alias("total_open_units"),
        F.sum(F.when(F.col("total_open_units") > 0, 1).otherwise(0)).alias("orders_with_open_units"),
        F.sum(F.col("is_overdue").cast("int")).alias("overdue_orders"),
        F.sum(F.when(F.col("total_ordered_units") == 0, 1).otherwise(0)).alias("orders_without_lines"),
    )
    .withColumn("as_of_time", F.to_timestamp(F.lit(AS_OF_TIME)))
)

region_target_cols = spark.table(GOLD_REGION_TABLE).columns
regional_gold_delta = regional_gold_delta.select(*region_target_cols)

region_target = DeltaTable.forName(spark, GOLD_REGION_TABLE)
(
    region_target.alias("t")
    .merge(regional_gold_delta.alias("s"), "t.region <=> s.region")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("Merged:", GOLD_REGION_TABLE)

# ------------------------------------------------------------
# 8. INCREMENTAL VALIDATION
# ------------------------------------------------------------
gold_lines = spark.table(GOLD_LINE_TABLE)
gold_orders = spark.table(GOLD_ORDER_TABLE)
gold_regions = spark.table(GOLD_REGION_TABLE)

# Validate affected keys only plus global uniqueness/invariants.
missing_affected_orders = (
    affected_order_ids
    .join(gold_orders.select("order_id"), "order_id", "left_anti")
    .count()
)

missing_affected_lines = (
    order_lines_affected.select("order_id", "line_id").distinct()
    .join(gold_lines.select("order_id", "line_id"), ["order_id", "line_id"], "left_anti")
    .count()
)

invalid_affected_lines = (
    gold_lines
    .join(affected_order_ids, "order_id", "inner")
    .filter(
        (F.col("open_units") < 0)
        | (F.col("capped_delivered_units") > F.col("ordered_quantity"))
    )
    .count()
)

duplicate_affected_lines = (
    gold_lines
    .join(affected_order_ids, "order_id", "inner")
    .groupBy("order_id", "line_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_affected_orders = (
    gold_orders
    .join(affected_order_ids, "order_id", "inner")
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks = {
    "affected_orders_present": missing_affected_orders == 0,
    "affected_lines_present": missing_affected_lines == 0,
    "affected_lines_have_valid_units": invalid_affected_lines == 0,
    "affected_line_keys_unique": duplicate_affected_lines == 0,
    "affected_order_keys_unique": duplicate_affected_orders == 0,
}

print("\n" + "=" * 70)
print("INCREMENTAL GOLD VALIDATION")
print("=" * 70)
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

failed = [name for name, passed in checks.items() if not passed]
if failed:
    raise RuntimeError("Incremental Gold validation failed: " + ", ".join(failed))

print("\nFINAL GOLD TABLE COUNTS")
print("order_line_fulfilment      :", gold_lines.count())
print("order_service_performance  :", gold_orders.count())
print("regional_backlog           :", gold_regions.count())

print("\nPHASE 5 INCREMENTAL GOLD JOB COMPLETED SUCCESSFULLY.")
print("Processed Silver batch:", BATCH_ID)
print("Affected orders:", affected_order_count)


PHASE 5 GOLD JOB STARTED
Catalog/schema : dbwteam3.order_fulfillment
Reporting as-of: 2026-10-01T00:00:00
Silver input checks passed.
Silver orders: 2040
Silver order lines: 4773
Silver delivery events: 6018
Silver pharmacies: 100
Written: dbwteam3.order_fulfillment.order_line_fulfilment
Written: dbwteam3.order_fulfillment.order_service_performance
Written: dbwteam3.order_fulfillment.regional_backlog

GOLD VALIDATION RESULTS
PASS: line_count_matches_silver
PASS: order_count_matches_silver
PASS: five_regional_rows
PASS: unique_order_line_keys
PASS: unique_order_keys
PASS: nonnegative_open_units
PASS: capped_delivery_not_above_ordered
PASS: all_orders_retained
PASS: cancelled_orders_excluded_from_otif
PASS: no_incomplete_order_passes_otif
PASS: no_late_line_order_passes_otif
PASS: regional_open_units_reconcile
PASS: regional_overdue_reconciles
PASS: regional_orders_without_lines_reconcile

FINAL TABLE COUNTS
order_line_fulfilment: 4773
order_service_performance: 2040
regional_backlog: 5
